In [5]:
import numpy as np
import os

# Program used to downscale the ground truth values without loosing any features
def downscale_depth_minpool(depth, target_h, target_w):
    H, W = depth.shape

    scale_y = H // target_h
    scale_x = W // target_w

    depth_ds = np.full((target_h, target_w), np.nan, dtype=np.float32)

    for y in range(target_h):
        for x in range(target_w):

            block = depth[
                y*scale_y:(y+1)*scale_y,
                x*scale_x:(x+1)*scale_x
            ]

            # keep only valid LiDAR pixels
            valid = block[np.isfinite(block)]

            if valid.size > 0:
                depth_ds[y, x] = np.min(valid)

    return depth_ds

In [8]:
# Path to npy ground truth file
ground_truth_data = "/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/output/all_depths.npy"
# Path to model dpeth truth file
model_depth_data = "/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/monodepth_v2/all_depth.npy"

# Loading stacks 
groundtruth_stack = np.load(ground_truth_data)
model_stack = np.load(model_depth_data)

print("Ground truth shape:", groundtruth_stack.shape)
print("Model shape:", model_stack.shape)

if groundtruth_stack.shape[0] != model_stack.shape[0]:
    print("Error in the total number of frames")
else:
    print("Frame counts match:", groundtruth_stack.shape[0])

target_h = model_stack.shape[1]
target_w = model_stack.shape[2]

resized_gt = []

for frame in groundtruth_stack:
    resized = downscale_depth_minpool(frame, target_h, target_w)
    resized_gt.append(resized)

resized_gt = np.stack(resized_gt)

print("Downscaled GT shape:", resized_gt.shape)

np.save("/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/monodepth_v2/groundtruth.npy", resized_gt)

Ground truth shape: (199, 1280, 1920)
Model shape: (199, 192, 640)
Frame counts match: 199
Downscaled GT shape: (199, 192, 640)
